# 01 — Classify a complaint

Full pipeline: ingest, guardrails/redaction, retrieval, classification,
review routing, evidence write, batch execution. Reflects ADR-0004
(2026-07-23): review routing uses retrieval-similarity delta rather than
the model's self-report, which was tested three ways and a fourth way
(token-level logprobs) and found not to discriminate on Granite 3.3 8B
Instruct. Also fixes a PII-counting bug present since the pipeline was
first built: `len()` on the guardrails response dict always returned 1
(one key, `"detections"`), regardless of actual content, making
`pii_detected` always `True` and `pii_redactions` always `1` on every
record all session. See ADR-0004 and ADR-0008 for full reasoning.

**Run this after:**
- The workbench is running with the taxonomy ConfigMap mounted
  (manifests/workbench/workbench.yaml)
- `ansible-playbook ansible/site.yml` has completed (fully automated: seeds
  data, restarts the predictor once weights are confirmed complete, restarts
  Llama Stack so it discovers the model)

## Environment variables required

| Variable | Description | Default |
|---|---|---|
| `LLAMA_STACK_URL` | Llama Stack service | `http://lsd-complaint-intelligence-service:8321` |
| `GUARDRAILS_URL` | Guardrails orchestrator, HTTPS, self-signed | `https://guardrails-orchestrator-service:8032` |
| `MINIO_ENDPOINT` | MinIO service | `minio.complaint-intelligence.svc.cluster.local:9000` |
| `MINIO_ACCESS_KEY` | MinIO access key | none, required |
| `MINIO_SECRET_KEY` | MinIO secret key | none, required |
| `TAXONOMY_PATH` | Mounted ConfigMap path | `/opt/app-root/taxonomy/taxonomy.yaml` |

## Review routing (ADR-0004, 2026-07-23)

A complaint routes to review if EITHER the model's own `confidence` falls
below `CONFIDENCE_THRESHOLD` (0.7, a coarse backstop that did not fire on
any tested record), OR the top two THEME-type taxonomy matches from vector
search are within `AMBIGUITY_DELTA_THRESHOLD` (0.03) of each other.
Demo-appropriate value, not a calibrated production threshold; see
ADR-0004 for the stated limitation.

## PII handling (fixed 2026-07-23)

Detected PII is now actually redacted, not just logged: `redacted_body`
replaces the raw complaint text everywhere downstream of Cell 7, sent to
the model, embedded in the vector store, and matched against for citation
verification. Population (Cell 15) now runs a guardrails check per
complaint before embedding, not just at classification time.

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install --quiet requests==2.32.3 pyyaml==6.0.2 minio==7.2.7
print("Dependencies installed.")
# Note: pip may warn about a urllib3 version conflict with appengine-python-standard,
# a package baked into the base workbench image and unrelated to this notebook.
# That warning is expected and harmless; only a raised exception here is a real failure.

## Cell 2 — Configuration

In [ ]:
import os

LLAMA_STACK_URL  = os.environ.get("LLAMA_STACK_URL", "http://lsd-complaint-intelligence-service:8321")
GUARDRAILS_URL   = os.environ.get("GUARDRAILS_URL", "https://guardrails-orchestrator-service:8032")
TAXONOMY_PATH    = os.environ.get("TAXONOMY_PATH", "/opt/app-root/taxonomy/taxonomy.yaml")

MINIO_ENDPOINT   = os.environ.get("MINIO_ENDPOINT", "minio.complaint-intelligence.svc.cluster.local:9000")
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]   # fail fast if not set
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]   # fail fast if not set

# NB: these are real bucket names, not the `mc` alias (`uc02`) used when
# seeding from a laptop. The alias is a local mc config convenience and is
# not part of the bucket name itself.
COMPLAINTS_BUCKET = "complaints"
EVIDENCE_BUCKET   = "evidence"
COMPLAINTS_KEY    = "incoming/records.jsonl"
EVIDENCE_PREFIX   = "classifications"

# Review-routing threshold, ADR-0004 (2026-07-23). Demo-appropriate value,
# set from two confirmed examples in the synthetic dataset, not calibrated
# against a statistically meaningful labeled set. See ADR-0004 for the
# stated limitation.
AMBIGUITY_DELTA_THRESHOLD = 0.03

import requests
print("Configuration loaded.")

## Cell 3 — PII detection and redaction helpers

Placed early, before anything else uses them: only depends on
`GUARDRAILS_URL` from Cell 2, nothing from the vector-store cells later
in this notebook. **Fixed 2026-07-23:** `check_pii()` previously measured
`len()` on the raw response dict (always `1`, one key: `"detections"`),
not the actual list inside it, which silently made `pii_detected` always
`True` and `pii_redactions` always `1` regardless of real content. Fixed
here by reading `response["detections"]` explicitly. `redact_pii()` uses
the validated response shape confirmed live 2026-07-21: `{"start", "end",
"text", "detection", "detection_type", "detector_id", "score"}`.

In [ ]:
def check_pii(text: str) -> tuple:
    """Calls the guardrails orchestrator, returns (pii_detected, spans)."""
    resp = requests.post(
        f"{GUARDRAILS_URL}/api/v2/text/detection/content",
        json={"detectors": {"regex": {"regex": ["email", "credit-card"]}}, "content": text},
        verify=False
    )
    resp.raise_for_status()
    spans = resp.json().get("detections", [])
    return len(spans) > 0, spans


def redact_pii(text: str, spans: list) -> str:
    """Masks detected PII spans. Processes spans in reverse start-offset
    order so earlier offsets are not shifted by replacements made later
    in the string."""
    for span in sorted(spans, key=lambda s: s["start"], reverse=True):
        placeholder = f"[REDACTED:{span['detection'].upper()}]"
        text = text[:span["start"]] + placeholder + text[span["end"]:]
    return text

print("PII helpers defined: check_pii, redact_pii")

## Cell 4 — Load taxonomy from the mounted ConfigMap

In [ ]:
import yaml
from pathlib import Path

taxonomy_file = Path(TAXONOMY_PATH)
if not taxonomy_file.exists():
    raise RuntimeError(
        f"Taxonomy not found at {TAXONOMY_PATH}. Confirm the taxonomy "
        f"ConfigMap volume is mounted (manifests/workbench/workbench.yaml)."
    )

taxonomy = yaml.safe_load(taxonomy_file.read_text())
TAXONOMY_VERSION = taxonomy["version"]
CONFIDENCE_THRESHOLD = taxonomy["classification"]["confidence_threshold"]

print(f"Taxonomy version {TAXONOMY_VERSION}: "
      f"{len(taxonomy['themes'])} themes, {len(taxonomy['root_causes'])} root causes")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

## Cell 5 — Discover the LLM at runtime

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/models")
resp.raise_for_status()
models = resp.json().get("data", resp.json())

llm_candidates = [m for m in models if "granite" in m.get("id", "").lower()]
if not llm_candidates:
    raise RuntimeError(
        f"No Granite model found in /v1/models. Got: {models}\n"
        f"If Granite's InferenceService is healthy but still missing here, "
        f"restart the lsd-complaint-intelligence pod and re-run this cell "
        f"(ansible/site.yml now does this automatically after seeding)."
    )

MODEL_ID = llm_candidates[0]["id"]
print(f"Using model: {MODEL_ID}")

## Cell 6 — Pull one complaint from MinIO

In [ ]:
from minio import Minio
import json

minio_client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                      secret_key=MINIO_SECRET_KEY, secure=False)

if not minio_client.bucket_exists(COMPLAINTS_BUCKET):
    raise RuntimeError(f"Bucket '{COMPLAINTS_BUCKET}' not found. Run ansible/site.yml.")

response = minio_client.get_object(COMPLAINTS_BUCKET, COMPLAINTS_KEY)
first_line = response.read().decode("utf-8").splitlines()[0]
response.close()
response.release_conn()

complaint = json.loads(first_line)
print(f"Loaded complaint {complaint['complaint_id']}")
print(complaint["body"][:300])

## Cell 7 — Guardrails check and redaction

`redacted_body` is what flows into the prompt, retrieval, and citation
matching from here on, not the raw text. This is the fix for the gap
where PII was detected and recorded but never actually removed from what
the model saw or what got embedded.

In [ ]:
pii_detected, pii_spans = check_pii(complaint["body"])
pii_redactions = len(pii_spans)
redacted_body = redact_pii(complaint["body"], pii_spans)

# Injection detector is not configured on this stack (manifests/guardrails/
# orchestrator-config.yaml only registers email/credit-card regex), so this
# is recorded honestly as not evaluated rather than assumed false.
injection_blocked = None

print(f"PII detected: {pii_detected} ({pii_redactions} matches)")
print(f"Injection check: not configured on this stack")
if pii_detected:
    print(f"Redacted body: {redacted_body[:300]}")

## Cell 8 — Retrieval route discovery

Confirms `/v1/vector_stores` and `/v1/files` are both present, needed
together since population attaches previously-uploaded file objects rather
than accepting raw text directly.

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/inspect/routes")
resp.raise_for_status()
routes = resp.json()
all_routes = routes.get("data", routes)

vector_routes = [r for r in all_routes if "vector" in str(r).lower()]
file_routes = [r for r in all_routes if r.get("route", "").startswith("/v1/files")]

if not vector_routes:
    raise RuntimeError("No vector routes found. Confirm ENABLE_INLINE_MILVUS (ADR-0008).")
if not file_routes:
    raise RuntimeError(
        "No /v1/files routes found. Vector store population needs a file "
        "upload step first (OpenAI Vector Stores convention)."
    )

print("Vector/retrieval routes:")
for r in vector_routes:
    print(" ", r)
print("\nFile routes:")
for r in file_routes:
    print(" ", r)

## Cell 9 — Discover the embedding model

Per ADR-0008: read at runtime, never hardcoded. The embedding model
changed between RHOAI 2.25 and 3.4, so treating it as a constant would
guarantee breakage on the next platform version.

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/models")
resp.raise_for_status()
models = resp.json().get("data", resp.json())

embedding_candidates = [m for m in models if m.get("custom_metadata", {}).get("model_type") == "embedding"]
if not embedding_candidates:
    raise RuntimeError(f"No embedding model found in /v1/models. Got: {models}")

embedding_model_id = embedding_candidates[0]["id"]
embedding_dimension = embedding_candidates[0]["custom_metadata"]["embedding_dimension"]
print(f"Embedding model: {embedding_model_id} ({embedding_dimension} dims)")

## Cell 10 — Create or find the vector store (idempotent)

In [ ]:
VECTOR_STORE_NAME = "uc02-complaint-intelligence"

list_resp = requests.get(f"{LLAMA_STACK_URL}/v1/vector_stores")
list_resp.raise_for_status()
existing = [s for s in list_resp.json().get("data", []) if s.get("name") == VECTOR_STORE_NAME]

if existing:
    VECTOR_STORE_ID = existing[0]["id"]
    print(f"Using existing store: {VECTOR_STORE_ID}")
else:
    create_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores",
        json={
            "name": VECTOR_STORE_NAME,
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
        }
    )
    if not create_resp.ok:
        raise RuntimeError(f"Store creation failed ({create_resp.status_code}): {create_resp.text}")
    VECTOR_STORE_ID = create_resp.json()["id"]
    print(f"Created store: {VECTOR_STORE_ID}")

## Cell 11 — Helpers: upload/attach, existing-document lookup, filtered search, ambiguity check

`existing_ids_for_kind()` paginates correctly via the `after` cursor
(ADR-0008: the listing endpoint defaults to 20 results per page and
silently ignores a `limit` override). `search_by_kind()` uses the
confirmed typed-filter shape (a flat dict silently returns zero results).
`taxonomy_theme_ambiguity()` implements the ADR-0004 review-routing
signal; token-level logprobs were also probed (2026-07-23) and ruled out,
probability mass sits at ~1.0 on the chosen theme token even on
known-ambiguous complaints.

In [ ]:
def add_document(text: str, metadata: dict) -> str:
    """Uploads text as a file, attaches it to the vector store with metadata.
    Returns the file_id."""
    upload_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/files",
        files={"file": ("doc.txt", text.encode("utf-8"))},
        data={"purpose": "assistants"}
    )
    if not upload_resp.ok:
        raise RuntimeError(f"File upload failed ({upload_resp.status_code}): {upload_resp.text}")
    file_id = upload_resp.json()["id"]

    attach_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/files",
        json={"file_id": file_id, "attributes": metadata}
    )
    if not attach_resp.ok:
        raise RuntimeError(f"Attach to store failed ({attach_resp.status_code}): {attach_resp.text}")
    return file_id


def existing_ids_for_kind(kind: str) -> set:
    """Returns the set of `id` attributes already attached for a given kind,
    so population loops can skip documents already present. Paginates using
    the `after` cursor (ADR-0008: this endpoint defaults to 20 results per
    page and does not honor a `limit` override, that param is silently
    ignored rather than erroring)."""
    ids = set()
    after = None
    while True:
        params = {"after": after} if after else {}
        resp = requests.get(f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/files", params=params)
        resp.raise_for_status()
        page = resp.json()
        files = page.get("data", [])
        ids.update(
            f["attributes"]["id"] for f in files
            if f.get("attributes", {}).get("kind") == kind and "id" in f.get("attributes", {})
        )
        if not page.get("has_more") or not files:
            break
        after = files[-1]["id"]
    return ids


def search_by_kind(query: str, kind: str, top_k: int = 3) -> list:
    """Runs a filtered search. Typed-filter syntax confirmed live 2026-07-22
    (a flat dict filter silently returns zero results rather than erroring,
    do not revert to that shape, see ADR-0008)."""
    resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/search",
        json={
            "query": query,
            "filters": {"type": "eq", "key": "kind", "value": kind},
            "max_num_results": top_k
        }
    )
    if not resp.ok:
        raise RuntimeError(f"Search failed ({resp.status_code}): {resp.text}")
    return resp.json().get("data", [])


def taxonomy_theme_ambiguity(body: str) -> dict:
    """ADR-0004 review-routing signal: uses retrieval similarity, independent
    of the LLM's self-report. If the top two THEME-type taxonomy matches are
    close in similarity, that is real signal from the embedding space.
    top_k=10 guarantees coverage of all 10 themes so the true second place
    is never missed to a root-cause document crowding it out."""
    hits = search_by_kind(body, "taxonomy", top_k=10)
    theme_hits = sorted(
        [h for h in hits if h["attributes"].get("item_type") == "theme"],
        key=lambda h: h["score"], reverse=True
    )
    if len(theme_hits) < 2:
        return {"top_theme": None, "top_score": None, "second_theme": None, "second_score": None, "delta": None}
    top, second = theme_hits[0], theme_hits[1]
    return {
        "top_theme": top["attributes"]["id"], "top_score": top["score"],
        "second_theme": second["attributes"]["id"], "second_score": second["score"],
        "delta": top["score"] - second["score"]
    }


def lookup_classification(complaint_id: str):
    """Looks up a complaint's confirmed classification from its evidence
    record, for label-enriched retrieval (2026-07-24): retrieved similar
    complaints were otherwise undifferentiated text, tokens without signal,
    since the model had no way to know how they were classified. Returns
    None if no evidence record exists yet, or if that record was routed to
    review: an unconfirmed classification is not the "human-confirmed"
    precedent the demo narrative depends on, so it is not surfaced as one."""
    try:
        obj = minio_client.get_object(EVIDENCE_BUCKET, f"{EVIDENCE_PREFIX}/{complaint_id}.json")
        rec = json.loads(obj.read())
        obj.close()
        obj.release_conn()
        if rec.get("routed_to_review"):
            return None
        return {"theme_id": rec["theme_id"], "root_cause_id": rec["root_cause_id"]}
    except Exception:
        return None

print("Helpers defined: add_document, existing_ids_for_kind, search_by_kind, taxonomy_theme_ambiguity")

## Cell 12 — Single-document round-trip checkpoint (validated 2026-07-22)

Confirms upload -> attach -> filtered search works end to end. Left in
place as a fast sanity check on future runs, not just a one-time gate.

In [ ]:
test_theme = taxonomy["themes"][0]
already_present = test_theme["id"] in existing_ids_for_kind("taxonomy")

if already_present:
    print(f"{test_theme['id']} already attached, skipping upload.")
else:
    test_text = f"{test_theme['name']}: {test_theme['definition'].strip()}"
    test_file_id = add_document(test_text, {"kind": "taxonomy", "item_type": "theme", "id": test_theme["id"]})
    print(f"Uploaded and attached: {test_file_id}")
    import time
    time.sleep(2)  # indexing may not be instant

results = search_by_kind(test_theme["definition"][:100], "taxonomy")
print(json.dumps(results, indent=2))
assert any(r["attributes"]["id"] == test_theme["id"] for r in results), \
    "Round-trip failed: expected document not found in filtered search results"
print("Round-trip confirmed: upload -> attach -> filtered search works.")

## Cell 13 — Populate the taxonomy (17 documents, idempotent)

One document per theme (10) and per root cause (7). Embeds `definition`,
`includes`, `excludes`, `examples` concatenated, matching taxonomy.yaml's
own stated intent that these fields are retrieval content, not decoration.
No PII expected in taxonomy text, so no redaction pass here.

In [ ]:
already = existing_ids_for_kind("taxonomy")
added = 0

for t in taxonomy["themes"]:
    if t["id"] in already:
        continue
    parts = [f"{t['name']}: {t['definition'].strip()}"]
    if t.get("includes"):
        parts.append("Includes: " + "; ".join(t["includes"]))
    if t.get("excludes"):
        parts.append("Excludes: " + "; ".join(t["excludes"]))
    if t.get("examples"):
        parts.append("Examples: " + "; ".join(t["examples"]))
    add_document("\n".join(parts), {"kind": "taxonomy", "item_type": "theme", "id": t["id"]})
    added += 1

for r in taxonomy["root_causes"]:
    if r["id"] in already:
        continue
    text = f"{r['name']}: {r['definition'].strip()}"
    add_document(text, {"kind": "taxonomy", "item_type": "root_cause", "id": r["id"]})
    added += 1

print(f"Added {added} new taxonomy documents ({len(already)} already present).")

import time
if added:
    time.sleep(3)

final_count = len(existing_ids_for_kind("taxonomy"))
expected = len(taxonomy["themes"]) + len(taxonomy["root_causes"])
print(f"Taxonomy documents in store: {final_count} (expected {expected})")
assert final_count == expected, "Taxonomy population incomplete, check for upload errors above."

## Cell 14 — Populate complaints (200 documents, idempotent, redacted)

**Changed 2026-07-23:** now runs a guardrails check and redaction pass on
each complaint BEFORE embedding, not just at classification time. This
closes the gap where detected PII was recorded in evidence but the raw
text still flowed into the vector store. Adds one guardrails call per
complaint (200 total) to this cell's runtime.

In [ ]:
response = minio_client.get_object(COMPLAINTS_BUCKET, COMPLAINTS_KEY)
all_lines = response.read().decode("utf-8").splitlines()
response.close()
response.release_conn()

all_complaints = [json.loads(line) for line in all_lines]
print(f"Loaded {len(all_complaints)} complaint records from MinIO.")

already = existing_ids_for_kind("complaint")
added = 0
redacted_count = 0

for c in all_complaints:
    if c["complaint_id"] in already:
        continue
    detected, spans = check_pii(c["body"])
    text_to_embed = redact_pii(c["body"], spans) if detected else c["body"]
    if detected:
        redacted_count += 1
    add_document(text_to_embed, {
        "kind": "complaint",
        "id": c["complaint_id"],
        "channel": c.get("channel", ""),
        "received_date": c.get("received_date", "")
    })
    added += 1
    if added % 25 == 0:
        print(f"  {added} uploaded...")

print(f"Added {added} new complaint documents ({len(already)} already present), "
      f"{redacted_count} contained PII and were redacted before embedding.")

import time
if added:
    time.sleep(5)

final_count = len(existing_ids_for_kind("complaint"))
print(f"Complaint documents in store: {final_count} (expected {len(all_complaints)})")
assert final_count == len(all_complaints), "Complaint population incomplete, check for upload errors above."

## Cell 15 — Retrieve context for the current complaint

Two filtered searches, combined into labeled sections. Excludes the
current complaint from its own retrieved context. Searches using
`redacted_body`, not raw text, consistent with what is now embedded in
the store.

**Changed 2026-07-24:** retrieved complaints are now labeled with their
confirmed classification (`lookup_classification()`, Cell 11), not shown
as bare undifferentiated text. This is genuine few-shot signal, "here is
a similar complaint and how it was classified", rather than three extra
paragraphs the model has no way to make use of. Complaints without a
confirmed classification yet (unclassified, or routed to review) are
labeled as such rather than silently treated as precedent.

In [ ]:
taxonomy_hits = search_by_kind(redacted_body, "taxonomy", top_k=3)
complaint_hits = [
    r for r in search_by_kind(redacted_body, "complaint", top_k=4)
    if r["attributes"].get("id") != complaint["complaint_id"]
][:3]

taxonomy_section = "\n".join(
    f"- {r['content'][0]['text']}" for r in taxonomy_hits
) if taxonomy_hits else "(none retrieved)"

complaint_lines = []
for r in complaint_hits:
    cid = r["attributes"].get("id")
    cls = lookup_classification(cid) if cid else None
    label = f" [confirmed classification: {cls['theme_id']}]" if cls else " [classification not yet confirmed]"
    complaint_lines.append(f"- {r['content'][0]['text'][:200]}{label}")
complaint_section = "\n".join(complaint_lines) if complaint_lines else "(none retrieved)"

retrieved_context = (
    f"Relevant taxonomy guidance:\n{taxonomy_section}\n\n"
    f"Similar past complaints:\n{complaint_section}"
)

print(retrieved_context)

## Cell 16 — Build the classification prompt

Uses `redacted_body`, not raw text: the model never sees detected PII.
Schema is the simple form (no self-reported ambiguity fields, ADR-0004
2026-07-23: tested and found not to discriminate on this model).

In [ ]:
theme_block = "\n".join(
    f"- {t['id']}: {t['name']} — {t['definition'].strip()}"
    for t in taxonomy["themes"]
)
root_cause_block = "\n".join(
    f"- {r['id']}: {r['name']} — {r['definition'].strip()}"
    for r in taxonomy["root_causes"]
)

prompt = f"""You are classifying a bank complaint against a fixed taxonomy.

Themes:
{theme_block}

Root causes:
{root_cause_block}

{retrieved_context}

Complaint:
\"\"\"{redacted_body}\"\"\"

Respond with JSON only, no other text, in this exact shape:
{{
  "theme_id": "THM-XX",
  "root_cause_id": "RC-XX",
  "confidence": 0.0,
  "citation_text": "the exact sentence from the complaint that supports this classification"
}}

citation_text must be copied verbatim from the complaint text above."""

print(prompt[:800], "...")

## Cell 17 — Call the model and parse the response

**Fixed 2026-07-23:** the model occasionally emits content after the
closing JSON brace (observed in 7/200 records during full-batch testing,
concentrated in harder/contested classifications). Strict `json.loads()`
rejects the whole response in that case ("Extra data"). Switched to
`json.JSONDecoder().raw_decode()`, which parses the first valid JSON
object and ignores anything trailing.

In [ ]:
completion_resp = requests.post(
    f"{LLAMA_STACK_URL}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 500,
        "temperature": 0.0
    }
)
completion_resp.raise_for_status()
raw = completion_resp.json()["choices"][0]["message"]["content"]

try:
    classification = json.JSONDecoder().raw_decode(raw.strip())[0]
except json.JSONDecodeError:
    raise RuntimeError(f"Model did not return valid JSON:\n{raw}")

print(json.dumps(classification, indent=2))

## Cell 18 — Locate the citation and apply review routing (ADR-0004)

Citation is matched against `redacted_body`, since that is what the model
actually saw; matching against the raw original would fail for any
citation touching a redacted span. Routes to review if confidence is
below threshold OR the retrieval-delta ambiguity check fires.

In [ ]:
citation_text = classification["citation_text"]
start = redacted_body.find(citation_text)

if start == -1:
    citation = {"start": None, "end": None, "text": citation_text}
    citation_verified = False
else:
    citation = {"start": start, "end": start + len(citation_text), "text": citation_text}
    citation_verified = True

confidence = classification["confidence"]

taxonomy_ambig = taxonomy_theme_ambiguity(redacted_body)

low_confidence = confidence < CONFIDENCE_THRESHOLD
narrow_margin = taxonomy_ambig["delta"] is not None and taxonomy_ambig["delta"] < AMBIGUITY_DELTA_THRESHOLD
routed_to_review = low_confidence or narrow_margin

reasons = []
if low_confidence:
    reasons.append(f"confidence {confidence:.2f} below threshold {CONFIDENCE_THRESHOLD}")
if narrow_margin:
    reasons.append(
        f"top two taxonomy matches ({taxonomy_ambig['top_theme']}: {taxonomy_ambig['top_score']:.2f}, "
        f"{taxonomy_ambig['second_theme']}: {taxonomy_ambig['second_score']:.2f}) "
        f"within {AMBIGUITY_DELTA_THRESHOLD} of each other"
    )
review_reason = "; ".join(reasons) if reasons else None

candidate_theme_ids = []
if narrow_margin:
    candidate_theme_ids = [{"theme_id": taxonomy_ambig["second_theme"], "score": taxonomy_ambig["second_score"]}]

print(f"Routed to review: {routed_to_review}")
print(f"Reason: {review_reason}")
print(f"Citation verified against redacted text: {citation_verified}")

## Cell 19 — Assemble and write the evidence record

In [ ]:
from datetime import datetime, timezone
import uuid
import io

evidence_record = {
    "complaint_id": complaint["complaint_id"],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "theme_id": classification["theme_id"],
    "root_cause_id": classification["root_cause_id"],
    "confidence": confidence,
    "citation": citation,
    "citation_verified": citation_verified,
    "routed_to_review": routed_to_review,
    "review_reason": review_reason,
    "candidate_themes": candidate_theme_ids,
    "pii_detected": pii_detected,
    "pii_redactions": pii_redactions,
    "injection_blocked": injection_blocked,
    "guardrail_policy_id": "regex",
    "prompt_version": "0.1.0",
    "model_version": MODEL_ID,
    "taxonomy_version": TAXONOMY_VERSION,
    "trace_id": completion_resp.json().get("id", str(uuid.uuid4()))
}

record_bytes = json.dumps(evidence_record).encode("utf-8")
record_key = f"{EVIDENCE_PREFIX}/{complaint['complaint_id']}.json"
minio_client.put_object(
    EVIDENCE_BUCKET, record_key,
    data=io.BytesIO(record_bytes),
    length=len(record_bytes),
    content_type="application/json"
)

print(f"Evidence record written to s3://{EVIDENCE_BUCKET}/{record_key}")
print(json.dumps(evidence_record, indent=2))

## Cell 20 — Batch: idempotency check

`already_classified()` checks whether an evidence record already exists,
so a batch run can resume rather than reprocess everything.

In [ ]:
def already_classified(complaint_id: str) -> bool:
    """Checks whether an evidence record already exists for this complaint.
    NOTE: the broad except treats ANY failure (auth, network, wrong bucket)
    as "not yet classified", not just a genuine missing-key case. Acceptable
    for a demo corpus checked immediately after each run; would need
    narrowing to the specific S3 NoSuchKey exception for production use."""
    try:
        minio_client.stat_object(EVIDENCE_BUCKET, f"{EVIDENCE_PREFIX}/{complaint_id}.json")
        return True
    except Exception:
        return False

print("already_classified() defined.")

## Cell 21 — Batch: classification logic as a self-contained function

Mirrors Cells 7, 15-19 exactly, including redaction and ADR-0004 review
routing. Self-contained (computes its own `theme_block`/`root_cause_block`,
uses its own parameter `c` throughout) rather than relying on globals left
over from the single-record cells above.

In [ ]:
def classify_complaint(c: dict) -> dict:
    """Runs one complaint through guardrails/redaction, retrieval,
    classification, and ADR-0004 review routing. Returns the evidence
    record; does not write it (see write_evidence_record)."""

    # Guardrails and redaction (Cell 7 logic, fixed 2026-07-23)
    pii_detected, pii_spans = check_pii(c["body"])
    pii_redactions = len(pii_spans)
    redacted_body = redact_pii(c["body"], pii_spans)
    injection_blocked = None  # not configured on this stack, see Cell 7

    # Retrieval context, against redacted text, labeled with confirmed
    # classifications where available (2026-07-24, matches Cell 15)
    taxonomy_hits = search_by_kind(redacted_body, "taxonomy", top_k=3)
    complaint_hits = [
        r for r in search_by_kind(redacted_body, "complaint", top_k=4)
        if r["attributes"].get("id") != c["complaint_id"]
    ][:3]
    taxonomy_section = "\n".join(f"- {r['content'][0]['text']}" for r in taxonomy_hits) if taxonomy_hits else "(none retrieved)"
    complaint_lines = []
    for r in complaint_hits:
        cid = r["attributes"].get("id")
        cls = lookup_classification(cid) if cid else None
        label = f" [confirmed classification: {cls['theme_id']}]" if cls else " [classification not yet confirmed]"
        complaint_lines.append(f"- {r['content'][0]['text'][:200]}{label}")
    complaint_section = "\n".join(complaint_lines) if complaint_lines else "(none retrieved)"
    retrieved_context = f"Relevant taxonomy guidance:\n{taxonomy_section}\n\nSimilar past complaints:\n{complaint_section}"

    # Prompt, against redacted text
    theme_block = "\n".join(f"- {t['id']}: {t['name']} — {t['definition'].strip()}" for t in taxonomy["themes"])
    root_cause_block = "\n".join(f"- {r['id']}: {r['name']} — {r['definition'].strip()}" for r in taxonomy["root_causes"])
    prompt = f"""You are classifying a bank complaint against a fixed taxonomy.

Themes:
{theme_block}

Root causes:
{root_cause_block}

{retrieved_context}

Complaint:
\"\"\"{redacted_body}\"\"\"

Respond with JSON only, no other text, in this exact shape:
{{
  "theme_id": "THM-XX",
  "root_cause_id": "RC-XX",
  "confidence": 0.0,
  "citation_text": "the exact sentence from the complaint that supports this classification"
}}

citation_text must be copied verbatim from the complaint text above."""

    # Model call
    completion_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/chat/completions",
        json={"model": MODEL_ID, "messages": [{"role": "user", "content": prompt}],
              "max_tokens": 500, "temperature": 0.0}
    )
    completion_resp.raise_for_status()
    raw = completion_resp.json()["choices"][0]["message"]["content"]
    try:
        classification = json.JSONDecoder().raw_decode(raw.strip())[0]
    except json.JSONDecodeError:
        raise RuntimeError(f"Model did not return valid JSON:\n{raw}")

    # Citation, matched against redacted text (what the model actually saw)
    citation_text = classification["citation_text"]
    start = redacted_body.find(citation_text)
    if start == -1:
        citation = {"start": None, "end": None, "text": citation_text}
        citation_verified = False
    else:
        citation = {"start": start, "end": start + len(citation_text), "text": citation_text}
        citation_verified = True

    # ADR-0004 review routing: confidence backstop OR retrieval-delta ambiguity
    confidence = classification["confidence"]
    taxonomy_ambig = taxonomy_theme_ambiguity(redacted_body)

    low_confidence = confidence < CONFIDENCE_THRESHOLD
    narrow_margin = taxonomy_ambig["delta"] is not None and taxonomy_ambig["delta"] < AMBIGUITY_DELTA_THRESHOLD
    routed_to_review = low_confidence or narrow_margin

    reasons = []
    if low_confidence:
        reasons.append(f"confidence {confidence:.2f} below threshold {CONFIDENCE_THRESHOLD}")
    if narrow_margin:
        reasons.append(
            f"top two taxonomy matches ({taxonomy_ambig['top_theme']}: {taxonomy_ambig['top_score']:.2f}, "
            f"{taxonomy_ambig['second_theme']}: {taxonomy_ambig['second_score']:.2f}) "
            f"within {AMBIGUITY_DELTA_THRESHOLD} of each other"
        )
    review_reason = "; ".join(reasons) if reasons else None

    candidate_theme_ids = []
    if narrow_margin:
        candidate_theme_ids = [{"theme_id": taxonomy_ambig["second_theme"], "score": taxonomy_ambig["second_score"]}]

    # Evidence assembly
    return {
        "complaint_id": c["complaint_id"],
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "theme_id": classification["theme_id"],
        "root_cause_id": classification["root_cause_id"],
        "confidence": confidence,
        "citation": citation,
        "citation_verified": citation_verified,
        "routed_to_review": routed_to_review,
        "review_reason": review_reason,
        "candidate_themes": candidate_theme_ids,
        "pii_detected": pii_detected,
        "pii_redactions": pii_redactions,
        "injection_blocked": injection_blocked,
        "guardrail_policy_id": "regex",
        "prompt_version": "0.1.0",
        "model_version": MODEL_ID,
        "taxonomy_version": TAXONOMY_VERSION,
        "trace_id": completion_resp.json().get("id", str(uuid.uuid4()))
    }


def write_evidence_record(record: dict) -> str:
    """Writes an evidence record to MinIO."""
    record_bytes = json.dumps(record).encode("utf-8")
    record_key = f"{EVIDENCE_PREFIX}/{record['complaint_id']}.json"
    minio_client.put_object(
        EVIDENCE_BUCKET, record_key,
        data=io.BytesIO(record_bytes),
        length=len(record_bytes),
        content_type="application/json"
    )
    return record_key

print("classify_complaint() and write_evidence_record() defined.")

## Cell 22 — Batch run over all 200 complaints

Skips already-classified complaints (resumable). Isolates per-record
failures so one bad record does not abort the run.

In [ ]:
response = minio_client.get_object(COMPLAINTS_BUCKET, COMPLAINTS_KEY)
all_lines = response.read().decode("utf-8").splitlines()
response.close()
response.release_conn()
all_complaints = [json.loads(line) for line in all_lines]

results = {"classified": [], "skipped": [], "failed": []}

for i, c in enumerate(all_complaints):
    if already_classified(c["complaint_id"]):
        results["skipped"].append(c["complaint_id"])
        continue
    try:
        record = classify_complaint(c)
        write_evidence_record(record)
        results["classified"].append(c["complaint_id"])
    except Exception as e:
        results["failed"].append({"complaint_id": c["complaint_id"], "error": str(e)})

    if (i + 1) % 25 == 0:
        print(f"  processed {i + 1}/{len(all_complaints)} "
              f"({len(results['classified'])} classified, "
              f"{len(results['skipped'])} skipped, "
              f"{len(results['failed'])} failed so far)")

print(f"\nDone. Classified: {len(results['classified'])}, "
      f"Skipped (already done): {len(results['skipped'])}, "
      f"Failed: {len(results['failed'])}")

routed_count = 0
pii_count = 0
for c in all_complaints:
    if c["complaint_id"] in [f["complaint_id"] for f in results["failed"]]:
        continue
    obj = minio_client.get_object(EVIDENCE_BUCKET, f"{EVIDENCE_PREFIX}/{c['complaint_id']}.json")
    rec = json.loads(obj.read())
    obj.close()
    obj.release_conn()
    if rec.get("routed_to_review"):
        routed_count += 1
    if rec.get("pii_detected"):
        pii_count += 1
print(f"Routed to review: {routed_count}/{len(all_complaints)}")
print(f"PII detected and redacted: {pii_count}/{len(all_complaints)}")

if results["failed"]:
    print("\nFailures:")
    for f in results["failed"]:
        print(f"  {f['complaint_id']}: {f['error']}")